In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from PIL import Image
import os
import numpy as np
from scipy.spatial.distance import mahalanobis

# --- 1. Custom Dataset für automatische Klassifizierung ---
class MotorDataset(Dataset):
    def __init__(self, main_dir, img_list, transform=None):
        self.main_dir = main_dir
        self.transform = transform
        self.all_imgs = img_list
        # Definition der gesunden Klassen
        self.class_names = ['S1000R_2015', 'S1000R_2017', 'S1000RR_2014']
        self.label_map = {name: i for i, name in enumerate(self.class_names)}

    def __len__(self):
        return len(self.all_imgs)

    def __getitem__(self, idx):
        img_loc = os.path.join(self.main_dir, self.all_imgs[idx])
        image = Image.open(img_loc).convert("RGB")
        filename = self.all_imgs[idx]
        label = -1
        for name in self.class_names:
            if name in filename:
                label = self.label_map[name]
                break
        if label == -1:
            raise ValueError(f"Datei {filename} konnte keiner Klasse zugeordnet werden!")
        if self.transform:
            image = self.transform(image)
        return image, label

# --- 2. Konfiguration & Chronologischer Split ---
dataset_path = '../data/processed/dataset_idle'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

data_transforms = transforms.Compose([
transforms.Resize((224, 224)),
    # 1. Bild-Manipulationen (auf PIL Image)
    transforms.ColorJitter(brightness=0.3, contrast=0.3), 
    
    # 2. Umwandlung in Tensor (JETZT hat das Bild eine .shape)
    transforms.ToTensor(),
    
    # 3. Tensor-Manipulationen
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)), # Simuliert "Löcher" im Spektrogramm/Datenverlust
    
    # 4. Normalisierung (immer als Letztes)
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Dateien laden und nach Quelle gruppieren für Anti-Leakage Split
all_filenames = [f for f in os.listdir(dataset_path) if f.endswith('.png') and "Youtube" not in f] # Nur eigene gesunde Daten

train_list = []
val_list = []
files_by_source = {}

for fname in all_filenames:
    source = fname.split('_idle_')[0]
    if source not in files_by_source:
        files_by_source[source] = []
    files_by_source[source].append(fname)

for source, images in files_by_source.items():
    images.sort(key=lambda x: float(x.split('_')[-1].replace('s.png', '')))
    split_idx = int(len(images) * 0.8)
    train_list.extend(images[0:split_idx])
    val_list.extend(images[split_idx:])

train_dataset = MotorDataset(dataset_path, train_list, transform=data_transforms)
val_dataset = MotorDataset(dataset_path, val_list, transform=data_transforms)

# Sampler für Balancing
train_labels = np.array([train_dataset[i][1] for i in range(len(train_dataset))])
class_sample_count = np.array([len(np.where(train_labels == t)[0]) for t in range(len(train_dataset.class_names))])
weight = 1. / class_sample_count
samples_weight = torch.from_numpy(np.array([weight[t] for t in train_labels])).double()
sampler = WeightedRandomSampler(samples_weight, len(samples_weight))

train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# --- 3. Modell Training (ResNet18) ---
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(train_dataset.class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Starte Training des Feature-Extraktors...")
for epoch in range(10):
    model.train()
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoche {epoch+1} abgeschlossen.")

# --- 4. Mahalanobis Anomalie-Logik ---
# Extrahiert den Vektor vor dem Fully Connected Layer
feature_extractor = torch.nn.Sequential(*(list(model.children())[:-1]))
feature_extractor.eval()

def get_features(loader):
    features = []
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            emb = feature_extractor(inputs)
            features.append(emb.cpu().view(emb.size(0), -1).numpy())
    return np.concatenate(features)

print("\nBerechne statistisches Profil der gesunden Motoren...")
healthy_features = get_features(train_loader)
mean_vec = np.mean(healthy_features, axis=0)
cov_matrix = np.cov(healthy_features, rowvar=False)
# Pseudo-Inverse für Stabilität bei korrelierten Features
inv_cov_matrix = np.linalg.pinv(cov_matrix) 



Starte Training des Feature-Extraktors...
Epoche 1 abgeschlossen.
Epoche 2 abgeschlossen.
Epoche 3 abgeschlossen.
Epoche 4 abgeschlossen.
Epoche 5 abgeschlossen.
Epoche 6 abgeschlossen.
Epoche 7 abgeschlossen.
Epoche 8 abgeschlossen.
Epoche 9 abgeschlossen.
Epoche 10 abgeschlossen.

Berechne statistisches Profil der gesunden Motoren...


In [21]:
def get_resnet_anomaly_score(img_input):
    # Prüfen, ob Pfad oder bereits geladenes PIL-Bild
    if isinstance(img_input, str):
        if not os.path.exists(img_input):
            return 0.0
        img_pil = Image.open(img_input).convert('RGB')
    else:
        img_pil = img_input
    
    input_tensor = data_transforms(img_pil).unsqueeze(0).to(device)
    
    with torch.no_grad():
        emb = feature_extractor(input_tensor).cpu().view(-1).numpy()
    
    # Berechne Mahalanobis-Distanz
    distance = mahalanobis(emb, mean_vec, inv_cov_matrix)
    return distance

In [22]:
# --- 5. Finale Inferenz (Pfade anpassen!) ---
print("\n--- Anomalie-Analyse ---")

# Beispielpfade (bitte prüfen/anpassen)
test_files = {
    "Gesund (S1000RR_2014)": '../data/processed/dataset_idle/S1000RR_2014_MaximilianHohmann_idle_45.25s.png',
    "Gesund (S1000R_2014)": '../data/processed/dataset_idle/S1000R_2014_YoutubeVToldsMotoShow_idle_1.75s.png',
    "Gesund (S1000RR_2012)": '../data/processed/dataset_idle/S1000RR_2012_YoutubeMartinNgim_idle_6.25s.png',
    "Defekt (Cam Chain 1)": '../data/processed/dataset_idle_defective/S1000R_201X_YoutubeBikeAndBuggy_CamChainRattle_idle_5.25s.png', 
    "Defekt (Cam Chain 1)": '../data/processed/dataset_idle_defective/S1000R_201X_YoutubeBikeAndBuggy_CamChainRattle_idle_9.25s.png', 
    "Defekt (Cam Chain 2)": '../data/processed/dataset_idle_defective/S1000RR_2014_YoutubeVasylSuprovych_CamChainRattle_idle_0.50s.png',
    "Defekt (Cam Chain 2)": '../data/processed/dataset_idle_defective/S1000RR_2014_YoutubeVasylSuprovych_CamChainRattle_idle_9.50s.png', 
}


for name, path in test_files.items():
    score = get_resnet_anomaly_score(path)
    if isinstance(score, float):
        print(f"{name:<20} | Score: {score:>8.4f}")
    else:
        print(score)


--- Anomalie-Analyse ---
Gesund (S1000RR_2014) | Score:  35.4114
Gesund (S1000R_2014) | Score: 122.8324
Gesund (S1000RR_2012) | Score: 180.3510
Defekt (Cam Chain 1) | Score: 179.6619
Defekt (Cam Chain 2) | Score: 225.1440


In [28]:
def get_contrast_ratio(img_path):
    img_pil = Image.open(img_path).convert('RGB')
    w, h = img_pil.size
    
    # Zerschneiden in Oben (Mechanik) und Unten (Basis)
    top_half = img_pil.crop((0, 0, w, h // 2))
    bottom_half = img_pil.crop((0, h // 2, w, h))
    
    # Scores berechnen
    score_top = get_resnet_anomaly_score(top_half)
    score_bottom = get_resnet_anomaly_score(bottom_half)
    
    # Ratio berechnen: Wie viel "kränker" ist die Mechanik als der Rest?
    ratio = score_top / score_bottom
    return ratio, score_top, score_bottom

# --- TEST ---
print(f"{'Datei':<25} | {'Ratio':<7} | {'Top':<8} | {'Bottom':<8}")
print("-" * 65)

test_files = [
    '../data/processed/dataset_idle/S1000RR_2014_MaximilianHohmann_idle_45.25s.png', # Gesund
    '../data/processed/dataset_idle/S1000R_2014_YoutubeVToldsMotoShow_idle_1.75s.png', # Gesund
    '../data/processed/dataset_idle/S1000R_2015_PhilipKehl_cold_idle_6.50s.png', # Gesund
    '../data/processed/dataset_idle_defective/S1000RR_2014_YoutubeVasylSuprovych_CamChainRattle_idle_9.50s.png', # Defekt
    '../data/processed/dataset_idle_defective/S1000R_201X_YoutubeBikeAndBuggy_CamChainRattle_idle_9.25s.png', # Defekt
]

for path in test_files:
    fname = os.path.basename(path)
    ratio, top, bottom = get_contrast_ratio(path)
    print(f"{fname[:25]:<25} | {ratio:>7.2f} | {top:>8.1f} | {bottom:>8.1f}")

Datei                     | Ratio   | Top      | Bottom  
-----------------------------------------------------------------
S1000RR_2014_MaximilianHo |    2.03 |    349.4 |    171.9
S1000R_2014_YoutubeVTolds |    1.75 |    231.8 |    132.7
S1000R_2015_PhilipKehl_co |    1.31 |    154.6 |    118.2
S1000RR_2014_YoutubeVasyl |    0.72 |    166.1 |    230.8
S1000R_201X_YoutubeBikeAn |    1.72 |    285.7 |    166.0
